# Lesson 5.5 — Architecture Documentation and Lifecycle Management

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Write a stakeholder-facing architecture document
- Manage SLA-based feedback loops and expectation alignment
- Plan lifecycle phases: discovery, design, handoff, monitoring, iteration

**You will produce:** An `architecture_doc.md` and a lifecycle/handoff plan.

## Concepts

An **architecture document** is the durable record that lets others build, operate, and evolve the system. It summarizes the problem, the design (stages, components, data flow), the decisions (linking ADRs), the SLAs, the risks/controls, and the operational runbook.

**Lifecycle management** (Domain 6) spans *discovery -> design -> handoff -> monitoring -> iteration*. Each phase has owners and exit criteria. **Handoff** is where projects fail silently: the receiving team needs the doc, the eval harness, the dashboards, the runbook, and the known limitations. **SLA-based feedback loops** keep expectations aligned: you report against the agreed metrics and feed production signals (Phase 3 observability, Phase 4 evals) back into iteration. This lesson assembles artifacts from every prior phase into the deliverable an organization owns.

> **Reference:** [Anthropic — Documentation home](https://docs.anthropic.com/en/home)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        return f"[MOCK {model}] would answer: {prompt[:80]}..."
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Assemble the architecture document

We generate a structured doc that references the artifacts produced across the course — ADR, architecture spec, guardrails, compliance, SLAs — so it is a real index, not prose.

In [ ]:
def architecture_doc(name, sla, adrs, controls, runbook):
    return "\n".join([
      f"# Architecture: {name}","",
      "## Overview\nInput -> processing (RAG + Claude) -> output (validated) -> feedback (eval + HITL).","",
      f"## SLAs\n{sla}","",
      "## Key decisions (ADRs)\n" + "\n".join(f"- {a}" for a in adrs),"",
      "## Controls\n" + "\n".join(f"- {c}" for c in controls),"",
      "## Runbook\n" + "\n".join(f"- {r}" for r in runbook),
    ])

doc = architecture_doc(
    "Prior-auth triage",
    sla="p95 < 20s; >=95% nurse agreement; 99.9% uptime",
    adrs=["ADR-001 workflow pattern","ADR-002 model tiering"],
    controls=["least-privilege tools","input/output guardrails","HIPAA matrix","HITL on high-impact"],
    runbook=["alert: retrieval-miss spike -> check index","alert: accuracy drop -> run eval set","rollback: pin prompt_version"],
)
with open("architecture_doc.md","w",encoding="utf-8") as fh: fh.write(doc)
print(doc)

## Code-Along: Step 2 — Write the lifecycle and handoff plan

We define each lifecycle phase with its owner, exit criteria, and handoff artifacts so the system can be operated by another team.

In [ ]:
lifecycle = [
  {"phase":"discovery","owner":"architect+PM","exit":"signed solution brief + SLAs"},
  {"phase":"design","owner":"architect","exit":"ADRs + architecture doc + eval plan"},
  {"phase":"handoff","owner":"architect->eng/ops","exit":"doc+harness+dashboards+runbook transferred"},
  {"phase":"monitoring","owner":"ops","exit":"alerts live; SLA report cadence set"},
  {"phase":"iteration","owner":"eng","exit":"A/B-validated improvements shipped"},
]
for p in lifecycle:
    print(f"{p['phase']:11} owner={p['owner']:16} exit: {p['exit']}")

## Exercise

Create a **handoff checklist** the receiving ops team signs: every item (architecture doc, eval harness, dashboards/alerts, runbook, risk register, compliance matrix, known limitations, on-call contacts) must be present and verified. Then write one sentence on how the SLA feedback loop drives the iteration phase.

In [ ]:
# TODO: handoff_checklist = [...]; mark each item verified True/False
# TODO: assert all verified before 'handoff complete'; note the SLA->iteration loop

## Key Takeaways

- The architecture doc indexes decisions, SLAs, controls, and the runbook for durability.
- Lifecycle = discovery -> design -> handoff -> monitoring -> iteration, each with exit criteria.
- Handoff fails silently without the doc, harness, dashboards, runbook, and known limits.
- SLA-based reporting closes the feedback loop and drives prioritized iteration.

---

**Next:** [Lesson 5.6 — Developer Productivity and Operational Enablement](./lesson-5.6-claude-code-enablement.ipynb)